In [1]:
pip install pandas tabulate rich


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import re
import json
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import nltk
from tabulate import tabulate
from rich.console import Console
from rich.table import Table

nltk.download("punkt")
console = Console()

# ==================================================
# LOAD DATA JSONL ARTIKEL
# ==================================================
jsonl_file = "tempo_articles.jsonl"
articles = []

with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            articles.append(json.loads(line))
        except json.JSONDecodeError:
            continue
 
# ==================================================
# FUNGSI BANTU
# ==================================================
def split_paragraphs(text):
    if not text:
        return []
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

# ==================================================
# PROSES DATASET ENTAILMENT
# ==================================================
output_file = "tempo_entailment_dataset_new.jsonl"

records_count = 0
premise_count = 0
hypothesis_sentence_count = 0

with open(output_file, "w", encoding="utf-8") as out_f:
    for idx, art in enumerate(tqdm(articles, desc="Memproses artikel")):
        doc_id = f"tempo_{idx+1:05d}"
        title = art.get("title", "")
        summary = art.get("summary", "")
        content = art.get("content", "")

        premise_paragraphs = split_paragraphs(summary)
        hypothesis_paragraphs = split_paragraphs(content)

        premise_count += len(premise_paragraphs)

        for p_id, premise in enumerate(premise_paragraphs, start=1):
            for h_id, hyp_paragraph in enumerate(hypothesis_paragraphs, start=1):
                hyp_sentences = sent_tokenize(hyp_paragraph)
                hypothesis_sentence_count += len(hyp_sentences)

                for hs_id, hypothesis in enumerate(hyp_sentences, start=1):
                    record = {
                        "doc_id": doc_id,
                        "title": title,
                        "paragraph_premise": p_id,
                        "paragraph_hypothesis": h_id,
                        "sentence_hypothesis": hs_id,
                        "premise": premise,
                        "hypothesis": hypothesis,
                        "content": content
                    }
                    json.dump(record, out_f, ensure_ascii=False)
                    out_f.write("\n")
                    records_count += 1

# ==================================================
# TAMPILAN STATISTIK RAPI (TABEL)
# ==================================================
stats_table = Table(title="Statistik Pembentukan Dataset Entailment")

stats_table.add_column("Metrik")
stats_table.add_column("Nilai", justify="right")

stats_table.add_row("Total Artikel", str(len(articles)))
stats_table.add_row("Total Premise (paragraf ringkasan)", str(premise_count))
stats_table.add_row("Total Kalimat Hypothesis", str(hypothesis_sentence_count))
stats_table.add_row("Total Pasangan Premise-Hypothesis", str(records_count))
stats_table.add_row(
    "Rata-rata pasangan per artikel",
    f"{records_count / len(articles):.2f}"
)

console.print(stats_table)

# ==================================================
# PREVIEW DATASET (AMAN MEMORY, TANPA LOAD FULL)
# ==================================================
from itertools import islice

preview_rows = []

with open(output_file, "r", encoding="utf-8") as f:
    for line in islice(f, 10):  # hanya ambil 10 baris pertama
        preview_rows.append(json.loads(line))

df_preview = pd.DataFrame(preview_rows)

preview_cols = [
    "doc_id",
    "paragraph_premise",
    "paragraph_hypothesis",
    "sentence_hypothesis",
    "premise",
    "hypothesis",
]

print("\nContoh 10 baris pertama dataset:\n")
print(
    tabulate(
        df_preview[preview_cols],
        headers="keys",
        tablefmt="fancy_grid",
        showindex=False,
        maxcolwidths=[12, 8, 8, 8, 40, 40]
    )
)
1

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\andik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Memproses artikel: 100%|██████████| 24772/24772 [01:57<00:00, 210.26it/s]


   Statistik Pembentukan Dataset Entailment    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metrik                             ┃  Nilai ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Total Artikel                      │  24772 │
│ Total Premise (paragraf ringkasan) │  24768 │
│ Total Kalimat Hypothesis           │ 585107 │
│ Total Pasangan Premise-Hypothesis  │ 585107 │
│ Rata-rata pasangan per artikel     │  23.62 │
└────────────────────────────────────┴────────┘


Contoh 10 baris pertama dataset:

╒═════════════╤═════════════════════╤════════════════════════╤═══════════════════════╤═══════════════════════════════════════╤══════════════════════════════════════════╕
│ doc_id      │   paragraph_premise │   paragraph_hypothesis │   sentence_hypothesis │ premise                               │ hypothesis                               │
╞═════════════╪═════════════════════╪════════════════════════╪═══════════════════════╪═══════════════════════════════════════╪══════════════════════════════════════════╡
│ tempo_00001 │                   1 │                      1 │                     1 │ Grand Egyptian Museum (GEM) dekat     │ MUSEUM Mesir Agung atau Grand Egyptian   │
│             │                     │                        │                       │ Piramida Giza resmi buka untuk publik │ Museum (GEM) bersiap menyambut           │
│             │                     │                        │                       │ pada 4 November 2025 setelah